# CSU steering — Colab runner
Thin runner: all code lives in `src/`. Loop: edit locally → commit → push → run cell 1 to pull → run a milestone cell.
Runtime → Change runtime type → GPU (L4 / A100 / T4).

In [ ]:
# 1. Get the latest code
import os
REPO = 'https://github.com/ntmphu/Predicting-Activation-Steering-Failure-from-CSU.git'
DIR = '/content/csu'
if not os.path.exists(DIR):
    !git clone -q {REPO} {DIR}
%cd {DIR}
!git pull -q && git log --oneline -1

In [ ]:
# 2. Install deps (Colab already has a CUDA torch; don't replace it)
!grep -v '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 3. Hugging Face backup (private dataset repo; no Google Drive / account needed)
# Paste a HF token with WRITE access. It stays in this session only, never in git.
import os
os.environ['PYTHONUNBUFFERED'] = '1'  # show script output (progress bars) live
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF token: ')
# restore earlier results, if any (fails harmlessly on the very first run)
!python -m src.hub pull || echo "nothing to restore yet"

In [ ]:
def backup():
    """Upload results/ to the HF repo. Run after every milestone."""
    !python -m src.hub push

## Milestones

In [ ]:
# Milestone 1: data split + prompt check
!python -m src.data --model Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
# Milestone 2a: decide the prompt format (copy the output into PROGRESS.md)
!python -m src.format_check --model Qwen/Qwen2.5-1.5B-Instruct --limit 100 --gen 30

In [ ]:
# Milestone 2b: record hidden states (after setting INSTRUCTION in src/config.py, push, re-run cell 1)
!python -m src.model --model Qwen/Qwen2.5-1.5B-Instruct
backup()

In [ ]:
# Milestone 3: steering hook checks (alpha=0 no-op, hook removal, sign)
!python -m src.steer --model Qwen/Qwen2.5-1.5B-Instruct --limit 20

In [ ]:
# Milestone 4: layer x strength sweep (~5 min on GPU), then show the plots
!python -m src.sweep --model Qwen/Qwen2.5-1.5B-Instruct --limit 100
from IPython.display import Image, display
for m in ['reply', 'all']: display(Image(f'results/figures/sweep_{m}.png'))
backup()

In [ ]:
# Milestone 4b: letter-confound diagnostic (plain vs letter-balanced vector, ~2 min)
!python -m src.diagnose --model Qwen/Qwen2.5-1.5B-Instruct --limit 100

In [ ]:
# Milestone 5: failure labels, 10 settings x 600 prompts (~5-10 min). Last GPU step.
!python -m src.labels --model Qwen/Qwen2.5-1.5B-Instruct
backup()

In [ ]:
# Milestones 6-8: CSU + baselines, AUROC table, risk-coverage plot (CPU only, ~2-5 min)
!python -m src.csu --model Qwen/Qwen2.5-1.5B-Instruct
!python -m src.analysis --n 25
from IPython.display import Image, display
display(Image('results/figures/risk_coverage_n25.png'))
backup()

## Extensions (Oct 27)
Run cells 1-4 first. Results now live in `results/<model>_<behavior>/`.

In [ ]:
# X0: move the first run (1.5B sycophancy) into its run folder (safe to re-run)
!python scripts/migrate_results.py

In [ ]:
# E1: construction-induced failures (CPU, existing data)
M15 = 'Qwen/Qwen2.5-1.5B-Instruct'
!python -m src.analysis --model {M15} --behavior sycophancy --label construction
backup()

In [ ]:
# E2: 8 random construction sets for each small n (GPU ~10-15 min), then CSU vs real variability
!python -m src.labels --model {M15} --behavior sycophancy --ns 10,16,25,50 --seeds 8 --tag e2
!python -m src.csu --model {M15} --behavior sycophancy --tag e2
!python -m src.variability --model {M15} --behavior sycophancy --tag e2
!python -m src.analysis --model {M15} --behavior sycophancy --tag e2
!python -m src.analysis --model {M15} --behavior sycophancy --tag e2 --label construction
backup()

In [ ]:
# Full pipeline for a new (model, behavior): M2 -> sweep (auto-picks layer/alpha) -> labels -> CSU -> analysis
def pipeline(model, behavior):
    !python -m src.model --model {model} --behavior {behavior}
    !python -m src.sweep --model {model} --behavior {behavior} --limit 100
    !python -m src.labels --model {model} --behavior {behavior}
    !python -m src.csu --model {model} --behavior {behavior}
    !python -m src.analysis --model {model} --behavior {behavior}
    !python -m src.analysis --model {model} --behavior {behavior} --label construction
    backup()

In [ ]:
# E3: Qwen2.5-3B, sycophancy (GPU ~20-30 min)
pipeline('Qwen/Qwen2.5-3B-Instruct', 'sycophancy')

In [ ]:
# E4: Qwen2.5-1.5B, myopic-reward (GPU ~10-15 min)
pipeline('Qwen/Qwen2.5-1.5B-Instruct', 'myopic-reward')

In [ ]:
# E5: gradient CSU (GPU ~2 min per run for the gradients, rest CPU). Run after E2 (and after E3/E4 for those).
# g(x) = dm/d(added vector): alpha * g^T v predicts the steering effect; gCSU = Var_b[g^T v_b]
def e5(model, behavior, tags=('',)):
    !python -m src.grad --model {model} --behavior {behavior}
    for t in tags:
        !python -m src.csu --model {model} --behavior {behavior} --tag "{t}"
        !python -m src.analysis --model {model} --behavior {behavior} --tag "{t}" --grad
        !python -m src.analysis --model {model} --behavior {behavior} --tag "{t}" --grad --label construction
    if 'e2' in tags:
        !python -m src.variability --model {model} --behavior {behavior} --tag e2
    backup()

e5('Qwen/Qwen2.5-1.5B-Instruct', 'sycophancy', tags=('e2', ''))
# after E3 / E4 finish:
# e5('Qwen/Qwen2.5-3B-Instruct', 'sycophancy')
# e5('Qwen/Qwen2.5-1.5B-Instruct', 'myopic-reward')
